# 20 Audio Encoder Comparison

- Embeddings are made one clip at a time (batch_size = 1)
- Batching changed them by up to 8%, so training embedding would not match the app's.

In [1]:
import time

import numpy as np

from woofdata.tasks import task_rows
from woofdecoder.branches.audio import encoder_from_config
from woofdecoder.config import load_config
from wooftrainer.trainers.embeddings import cache_embeddings, load_embeddings
from wooftrainer.torch_datasets import TaskDataset

## Loading necessary project configurations


In [2]:
encoder = encoder_from_config(load_config())
print(f"encoder name: {encoder.name}")
print(f"encoder device: {encoder.device}")
print(f"encoder num_layers: {encoder.num_layers}")
print(f"encoder dim: {encoder.dim}")
print(f"encoder settings: {encoder.settings}")

c:\Users\kimy8\Desktop\Deep Learning\dog-vocal-interpreter\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


encoder name: beats_iter3_plus_as2m
encoder device: cuda
encoder num_layers: 12
encoder dim: 768
encoder settings: EncoderSettings(min_seconds=1.0, chunk_seconds=10.0, batch_size=1)


## Caching resulting embeddings

In [3]:
SPLITS = [
    ("audio_emotion", "val"),
    ("audio_emotion", "train"),
    ("audio_context", "val"),
    ("audio_context", "test"),
    ("audio_context", "train")
]

for task, split in SPLITS:
    start = time.perf_counter()
    
    cache_embeddings(
        task=task,
        split=split,
        encoder=encoder
    )
    
    print(f"Task: {task}, Split: {split} took {time.perf_counter() - start:.0f} s")

audio_emotion val: 200 clips, 0 already done
Task: audio_emotion, Split: val took 7 s
audio_emotion train: 1000 clips, 0 already done
500/1000
1000/1000
Task: audio_emotion, Split: train took 34 s
audio_context val: 403 clips, 0 already done
Task: audio_context, Split: val took 30 s
Reminder: test is reported once, at the end. Tune on val.
audio_context test: 357 clips, 0 already done
Task: audio_context, Split: test took 27 s
audio_context train: 12980 clips, 0 already done
500/12980
1000/12980
1500/12980
2000/12980
2500/12980
3000/12980
3500/12980
4000/12980
4500/12980
5000/12980
5500/12980
6000/12980
6500/12980
7000/12980
7500/12980
8000/12980
8500/12980
9000/12980
9500/12980
10000/12980
10500/12980
11000/12980
11500/12980
12000/12980
12500/12980
Task: audio_context, Split: train took 549 s


### Check if cache match the manifest


In [5]:
for task, split in SPLITS:
    embeddings, table = load_embeddings(task, split, encoder.name)
    rows = task_rows(task, split)
    same_clips = len(rows) == len(table) and set(rows["clip_id"]) == set(table["clip_id"])
    
    print(f"Task: {task:<14} Split: {split:<5} Shape: {embeddings.shape}")
    print(f"Matches Manifest: {same_clips} Finite: {np.isfinite(embeddings).all()}")

Task: audio_emotion  Split: val   Shape: (200, 12, 768)
Matches Manifest: True Finite: True
Task: audio_emotion  Split: train Shape: (1000, 12, 768)
Matches Manifest: True Finite: True
Task: audio_context  Split: val   Shape: (403, 12, 768)
Matches Manifest: True Finite: True
Reminder: test is reported once, at the end. Tune on val.
Task: audio_context  Split: test  Shape: (357, 12, 768)
Matches Manifest: True Finite: True
Task: audio_context  Split: train Shape: (12980, 12, 768)
Matches Manifest: True Finite: True


### Check labels

In [7]:
_, emotion = load_embeddings("audio_emotion", "train", encoder.name)
print(emotion["arousal"].value_counts(), "\n")
print(emotion["valence"].value_counts(), "\n")

_, context = load_embeddings("audio_context", "train", encoder.name)
print(context["mapped_category"].replace("", "(none)").value_counts(), "\n")

print("manifest columns:", list(task_rows("audio_emotion", "train").columns))

arousal
Medium    366
High      323
Low       311
Name: count, dtype: int64 

valence
Positive    429
Neutral     340
Negative    231
Name: count, dtype: int64 

mapped_category
playful_excited      5570
other                3157
attention_seeking    2429
alert_aggressive     1824
Name: count, dtype: int64 

manifest columns: ['clip_id', 'source', 'source_version', 'source_license', 'dog_id', 'leakage_group', 'split', 'modality', 'file_path', 'start_s', 'end_s', 'target_path', 'annotation_path', 'original_label', 'label_type', 'label_provenance', 'mapped_category', 'eval_allowed', 'notes']
